# Fine‑tuning de Qwen3 1.7B Instruct en Colab (GPU T4)

Fine‑tune de `unsloth/Qwen3-1.7B-Instruct` (variante **instruct**, no el base;
ver la conclusión de `documentacion/llm/base-vs-instruct.md`) en QLoRA con el
dataset real de ClimaSafeAI
(300 train / 100 val) y los hiperparámetros de `climasafeai/llm/fine_tune.py`.

**Antes de empezar:**

1. Menú **Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU**.
2. Ejecuta las celdas en orden. Cuando una te pida subir un fichero
   (`colab_dataset.zip`, `fine_tune.py`), usa el botón de subida que aparece —
   o deja los ficheros en `MyDrive/climasafeai/` y la celda los cogerá de ahí.

**Al terminar, los artefactos quedan en `MyDrive/climasafeai/`:**

- `qwen3-climasafe-lora/` → adaptador LoRA (sobre el base instruct)
- `qwen3-climasafe-q4_k_m.gguf` → modelo cuantizado q4_k_m

## 1. Comprobar la GPU

Unsloth y el QLoRA no funcionan en CPU. Esta celda aborta con un mensaje claro
si el runtime no tiene GPU activada.

In [ ]:
import shutil
import subprocess
import sys


def gpu_detectada() -> str | None:
    # nvidia-smi es la forma directa de saber si el runtime tiene GPU
    if shutil.which("nvidia-smi"):
        salida = subprocess.run(
            ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
            capture_output=True, text=True,
        )
        if salida.returncode == 0 and salida.stdout.strip():
            return salida.stdout.strip().splitlines()[0]
    # fallback: torch, si ya está instalado
    try:
        import torch
        if torch.cuda.is_available():
            return torch.cuda.get_device_name(0)
    except ImportError:
        pass
    return None


nombre = gpu_detectada()
if nombre is None:
    sys.exit(
        "NO hay GPU en este runtime. Unsloth y el QLoRA NO funcionan en CPU.\n"
        "Activa la GPU: Entorno de ejecución → Cambiar tipo de entorno de "
        "ejecución → T4 GPU, y vuelve a ejecutar esta celda."
    )
print(f"GPU detectada: {nombre}")

## 2. Instalar Unsloth

Receta oficial de Unsloth para Colab: `unsloth[colab-new]` (NO la de
`climasafeai/llm/entrenar.sh`, que fuerza torch 2.5.1 y rompe la importación
de unsloth con el runtime de Colab 2026: el torchao nuevo usa `torch.int1`,
que solo existe desde torch 2.6). Se desinstala el stack viejo antes de
instalar para que la sesión no arrastre la mezcla.

> `timm` y `fastai` son preinstalados de Colab que dependen de `torchvision`
> y no se usan en este flujo (cero menciones en `climasafeai/`): por eso
> también se desinstalan, para que no arrastren una versión vieja de torch.

In [ ]:
# Receta oficial de Unsloth para Colab: NO forzar torch viejo.
# El runtime de Colab 2026 ya trae torch + torchao recientes; instalar torch 2.5.1
# encima rompe la importación de unsloth (torchao usa torch.int1, que solo existe
# desde torch 2.6). unsloth[colab-new] respeta la CUDA del runtime.
!pip uninstall -y torch torchvision torchaudio xformers transformers unsloth torchao trl peft datasets accelerate bitsandbytes timm fastai

!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

## 3. Montar Google Drive

El adaptador LoRA y el GGUF se guardan en Drive para poder bajarlos después
desde cualquier máquina.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

## 4. Subir y verificar el dataset

Sube el `colab_dataset.zip` que generaste en local con
`climasafeai/llm/empaquetar_dataset_colab.py` (o déjalo en
`MyDrive/climasafeai/colab_dataset.zip`).

La celda verifica que es la **versión buena**, no la fake:

- `train.jsonl` con **300** líneas y `val.jsonl` con **100**.
- El campo **"Tiempo en esa franja"** presente en **todos** los inputs
  (la versión fake del `_predecir_fake` no lo tiene).
- Comprueba el **sha256** de ambos contra el del dataset **regenerado en LLM-017**
  (`train.jsonl` 300, `val.jsonl` 100) — el zip debe ser el re-empaquetado,
  no el de LLM-006.

In [ ]:
import hashlib
import json
import zipfile
from pathlib import Path

DATA_DIR = Path("/content/data")
DRIVE_DIR = Path("/content/drive/MyDrive/climasafeai")
DRIVE_ZIP = DRIVE_DIR / "colab_dataset.zip"

EXPECT = {"train.jsonl": 300, "val.jsonl": 100}
# sha256 del dataset regenerado en LLM-017 (re-empaquetado en colab_dataset.zip)
EXPECT_SHA256 = {
    "train.jsonl": "2566f1ecb9cae17bac1b77ac6821535bf8cc2bb677f407ebe1a239158f32d017",
    "val.jsonl": "865e7e5a49035cacc1c6e30352abe06d91a1627b79e27dab040e7b755bf46784",
}
MARCA = "Tiempo en esa franja"


def sha256(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 16), b""):
            h.update(chunk)
    return h.hexdigest()


# 1) Localizar el zip: primero en Drive, si no se pide subirlo
zip_path = None
if DRIVE_ZIP.exists():
    zip_path = DRIVE_ZIP
    print(f"Zip encontrado en Drive: {DRIVE_ZIP}")
else:
    from google.colab import files

    print("Sube el fichero colab_dataset.zip (el del empaquetado local):")
    subidos = files.upload()
    for nombre in subidos:
        if nombre.endswith(".zip"):
            zip_path = Path(nombre)
if zip_path is None:
    raise SystemExit(
        "No hay colab_dataset.zip. Súbelo con el botón o cópialo a "
        "MyDrive/climasafeai/colab_dataset.zip y repite esta celda."
    )

# 2) Descomprimir en /content/data
DATA_DIR.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(zip_path) as z:
    z.extractall(DATA_DIR)
print("Contenido de /content/data:", sorted(p.name for p in DATA_DIR.iterdir()))

# 3) Verificar la versión buena: 300/100 + marca en todos los inputs
errores = []
for nombre, esperadas in EXPECT.items():
    fichero = DATA_DIR / nombre
    ejemplos = []
    with open(fichero) as f:
        for linea in f:
            if linea.strip():
                ejemplos.append(json.loads(linea))
    con_marca = sum(1 for e in ejemplos if MARCA in (e.get("input") or ""))
    digest = sha256(fichero)
    print(f"  {nombre}: {len(ejemplos)} líneas, {con_marca} con '{MARCA}' "
          f"— sha256 {digest}")
    if len(ejemplos) != esperadas:
        errores.append(f"{nombre}: {len(ejemplos)} líneas (esperadas {esperadas})")
    if digest != EXPECT_SHA256[nombre]:
        errores.append(
            f"{nombre}: sha256 {digest} no coincide con el del dataset "
            f"regenerado en LLM-017 ({EXPECT_SHA256[nombre]}) — ¿es el zip viejo?"
        )
    if con_marca != len(ejemplos):
        errores.append(
            f"{nombre}: solo {con_marca}/{len(ejemplos)} inputs tienen "
            f"'{MARCA}' — ¿es la versión fake?"
        )

if errores:
    print("\nEl dataset NO es la versión buena:")
    for e in errores:
        print(f"  - {e}")
    print("Vuelve a empaquetar en local y repite esta celda.")
    raise SystemExit(1)
print("\nDataset OK: versión buena (300/100) verificada.")

## 5. Entrenar (QLoRA)

`fine_tune.py` no viene con el notebook: sube
`climasafeai/llm/fine_tune.py` (o déjalo en `MyDrive/climasafeai/fine_tune.py`)
y la celda lo copia a `/content/`.

El LoRA parte de la variante **instruct** `unsloth/Qwen3-1.7B-Instruct` (no del
base), según la conclusión de `documentacion/llm/base-vs-instruct.md`: el
adaptador solo tiene que enseñar el dominio ClimaSafe, no el comportamiento de
asistente. El `--model` que se pasa a `fine_tune.py` debe resolver a esa variante.

Hiperparámetros idénticos a los de `fine_tune.py`: rank 16, seq 1024, batch 2 ×
accum 4, 3 épocas, lr 2e-4. Con 300 ejemplos (~178k tokens) en la T4 son del
orden de minutos.

El LoRA se guarda directamente en Drive: `MyDrive/climasafeai/qwen3-climasafe-lora/`.

In [ ]:
import shutil
from pathlib import Path

SCRIPT = Path("/content/fine_tune.py")
DRIVE_SCRIPT = Path("/content/drive/MyDrive/climasafeai/fine_tune.py")

# fine_tune.py se sube o se copia desde Drive; nunca desde paths locales
if not SCRIPT.exists():
    if DRIVE_SCRIPT.exists():
        shutil.copy(DRIVE_SCRIPT, SCRIPT)
        print(f"Copiado fine_tune.py desde {DRIVE_SCRIPT}")
    else:
        from google.colab import files

        print("Sube climasafeai/llm/fine_tune.py (o cópialo a "
              "MyDrive/climasafeai/fine_tune.py):")
        subidos = files.upload()
        if "fine_tune.py" not in subidos:
            raise SystemExit("No se subió fine_tune.py")
print("fine_tune.py listo:", SCRIPT)

# Entrenamiento QLoRA con los hiperparámetros de fine_tune.py
!python /content/fine_tune.py \
  --model qwen3-1.7b \
  --train-file /content/data/train.jsonl \
  --val-file /content/data/val.jsonl \
  --output-dir /content/drive/MyDrive/climasafeai/qwen3-climasafe-lora \
  --epochs 3 \
  --max-seq-len 1024 \
  --lora-rank 16 \
  --batch-size 2 \
  --gradient-accum 4 \
  --lr 2e-4

## 6. Exportar a GGUF (q4_k_m)

Fusiona el LoRA con el modelo base y exporta a GGUF **q4_k_m** en Drive.

Es el paso que más VRAM pide (carga el modelo en fp16 para fusionar): correlo
en esta celda aparte, con la VRAM liberada. El 1.7B cabe en la T4 (16 GB).

In [ ]:
# Exportar LoRA → GGUF q4_k_m, directamente a Drive
!python /content/fine_tune.py --export-gguf \
  --lora-path /content/drive/MyDrive/climasafeai/qwen3-climasafe-lora \
  --gguf-path /content/drive/MyDrive/climasafeai/qwen3-climasafe-q4_k_m.gguf

## 7. Verificar los artefactos en Drive

Confirma que el LoRA y el GGUF están persistidos en `MyDrive/climasafeai/`.
Ya puedes bajarlos a tu máquina (ver `documentacion/llm/colab-fine-tuning.md`).

> **Qwen3 y el modo thinking:** el fine-tune entrena respuestas directas (formato
> EXACTO del benchmark), pero Qwen3 piensa por defecto (bloque `<think>`). Al crear
> el modelo en Ollama con el Modelfile de qwen3 hay que dejar el thinking
> desactivado (`enable_thinking=false` en el template de Ollama) para que el modelo
> no razone y rompa el formato `RIESGO:` / `Índice personalizado`.

In [ ]:
from pathlib import Path

DRIVE_DIR = Path("/content/drive/MyDrive/climasafeai")
lora = DRIVE_DIR / "qwen3-climasafe-lora"
gguf = DRIVE_DIR / "qwen3-climasafe-q4_k_m.gguf"

print("Artefactos en Drive:")
print(f"  LoRA:   {lora}/")
print(f"          {len(list(lora.glob('*')))} ficheros "
      f"(adapter_config.json, adapter_model.safetensors, ...)")
print(f"  GGUF:   {gguf}  ({gguf.stat().st_size / 1024**3:.2f} GB)")
print("\nListo: baja ambos artefactos a tu máquina y sirve con Ollama.")